<a href="https://colab.research.google.com/github/rachmi00/Traffic-Sign-Recognition/blob/main/Traffic_Sign_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

PERSISTENT_ROOT = Path('/content/drive/My Drive/TrafficSignProject')
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

# Update DATASET_DIR to use Drive
DATASET_DIR = PERSISTENT_ROOT / "dataset"
print(f"Data will now be stored persistently at: {DATASET_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data will now be stored persistently at: /content/drive/My Drive/TrafficSignProject/dataset


In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub
path = kagglehub.dataset_download("ferrantealessandro/street-sign-set")
print(path)

Using Colab cache for faster access to the 'street-sign-set' dataset.
/kaggle/input/street-sign-set


In [ ]:
import os
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.yaml'):
            print(os.path.join(root, file))

/kaggle/input/street-sign-set/StreetSignSet/data.yaml


In [ ]:
import yaml
# The file is located in 'StreetSignSet' rather than 'dataset'
with open(f"{path}/StreetSignSet/data.yaml") as f:
    ss_config = yaml.safe_load(f)
print(ss_config['names'])

ss_names = ss_config['names']
print(f"\nStreetSignSet has {len(ss_names)} classes.")

['prio_give_way', 'prio_stop', 'prio_priority_road', 'forb_speed_over_5', 'forb_speed_over_10', 'forb_speed_over_20', 'forb_speed_over_30', 'forb_speed_over_40', 'forb_speed_over_50', 'forb_speed_over_60', 'forb_speed_over_70', 'forb_speed_over_80', 'forb_speed_over_90', 'forb_speed_over_100', 'forb_speed_over_110', 'forb_speed_over_120', 'forb_speed_over_130', 'forb_no_entry', 'forb_no_parking', 'forb_no_stopping', 'forb_overtake_car', 'forb_overtake_trucks', 'forb_trucks', 'forb_turn_left', 'forb_turn_right', 'forb_weight_over_3.5t', 'forb_weight_over_7.5t', 'forb_u_turn', 'info_bus_station', 'info_crosswalk', 'info_highway', 'info_one_way', 'info_parking', 'info_taxi_parking', 'warn_children', 'warn_construction', 'warn_crosswalk', 'warn_cyclists', 'warn_left_curve', 'warn_right_curve', 'warn_domestic_animals', 'warn_other_dangers', 'warn_poor_road_surface', 'warn_roundabout', 'warn_sharp_left_curve', 'warn_sharp_right_curve', 'warn_slippery_road', 'warn_hump', 'warn_traffic_light',

In [ ]:
STREETSIGN_TO_OUR_CLASS = {
    "forb_speed_over_30":  0,   # Speed_Limit_30
    "forb_speed_over_50":  1,   # Speed_Limit_50
    "prio_priority_road":  2,   # Priority_Road
    "prio_give_way":       3,   # Give_Way
    "prio_stop":           4,   # Stop
    "forb_no_entry":       5,   # No_Entry
    "warn_construction":   6,   # Road_Work
    "warn_traffic_light":  7,   # Traffic_Lights_Ahead
    "warn_crosswalk":      8,   # Pedestrian_Crossing
    "warn_roundabout":     9,   # Roundabout
    "forb_no_parking":    10,   # No_Parking
}

OUR_CLASS_NAMES = [
    "Speed_Limit_30", "Speed_Limit_50", "Priority_Road", "Give_Way", "Stop",
    "No_Entry", "Road_Work", "Traffic_Lights_Ahead", "Pedestrian_Crossing", "Roundabout",
    "No_Parking",
]

In [ ]:
#Build StreetSignSet's own internal index → our class ID
# (StreetSignSet's data.yaml gives us names in order: index 0 = ss_names[0], etc.)
ss_index_to_our_class = {}
for ss_index, ss_name in enumerate(ss_names):
    if ss_name in STREETSIGN_TO_OUR_CLASS:
        ss_index_to_our_class[ss_index] = STREETSIGN_TO_OUR_CLASS[ss_name]

print(f"\nMatched {len(ss_index_to_our_class)} of {len(STREETSIGN_TO_OUR_CLASS)} target classes:")
for ss_index, our_id in ss_index_to_our_class.items():
    print(f"  StreetSignSet[{ss_index}] '{ss_names[ss_index]}'  →  our class {our_id} ({OUR_CLASS_NAMES[our_id]})")

missing = set(STREETSIGN_TO_OUR_CLASS.keys()) - {ss_names[i] for i in ss_index_to_our_class}
if missing:
    print(f"\n[!] WARNING — these expected classes were NOT found in StreetSignSet's data.yaml: {missing}")
    print("    Check for spelling differences and adjust STREETSIGN_TO_OUR_CLASS above.")


Matched 11 of 11 target classes:
  StreetSignSet[0] 'prio_give_way'  →  our class 3 (Give_Way)
  StreetSignSet[1] 'prio_stop'  →  our class 4 (Stop)
  StreetSignSet[2] 'prio_priority_road'  →  our class 2 (Priority_Road)
  StreetSignSet[6] 'forb_speed_over_30'  →  our class 0 (Speed_Limit_30)
  StreetSignSet[8] 'forb_speed_over_50'  →  our class 1 (Speed_Limit_50)
  StreetSignSet[17] 'forb_no_entry'  →  our class 5 (No_Entry)
  StreetSignSet[18] 'forb_no_parking'  →  our class 10 (No_Parking)
  StreetSignSet[35] 'warn_construction'  →  our class 6 (Road_Work)
  StreetSignSet[36] 'warn_crosswalk'  →  our class 8 (Pedestrian_Crossing)
  StreetSignSet[43] 'warn_roundabout'  →  our class 9 (Roundabout)
  StreetSignSet[48] 'warn_traffic_light'  →  our class 7 (Traffic_Lights_Ahead)


In [ ]:
import shutil
from pathlib import Path

# 1. Setup paths
KAG_DIR = Path(path) / "StreetSignSet"
# Use the persistent path from your Drive mount cell
DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')

# Mapping for original splits
SPLIT_MAP = {"train": "train", "val": "valid", "test": "valid"}

print(f"Processing raw data from Kaggle into {DATASET_DIR}...")

for src_split, dst_split in SPLIT_MAP.items():
    src_img_dir = KAG_DIR / src_split / "images"
    src_lbl_dir = KAG_DIR / src_split / "labels"

    if not src_img_dir.exists():
        continue

    out_img_dir = DATASET_DIR / dst_split / "images"
    out_lbl_dir = DATASET_DIR / dst_split / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    for lbl_path in src_lbl_dir.glob("*.txt"):
        kept_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                ss_idx = int(parts[0])
                if ss_idx in ss_index_to_our_class:
                    our_id = ss_index_to_our_class[ss_idx]
                    kept_lines.append(f"{our_id} {' '.join(parts[1:])}")

        if kept_lines:
            img_path = None
            for ext in (".jpg", ".jpeg", ".png"):
                candidate = src_img_dir / (lbl_path.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if img_path:
                shutil.copy2(img_path, out_img_dir / img_path.name)
                (out_lbl_dir / lbl_path.name).write_text("\n".join(kept_lines))

print(f"Done! Persistent dataset at {DATASET_DIR} is now populated with all classes.")

Processing raw data from Kaggle into /content/drive/My Drive/TrafficSignProject/dataset...
Done! Persistent dataset at /content/drive/My Drive/TrafficSignProject/dataset is now populated with all classes.


In [ ]:
import shutil
import random
from pathlib import Path
from collections import Counter

# Pointing back to your persistent Google Drive folder
DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')
TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "valid"

OUR_CLASS_NAMES = [
    "Speed_Limit_30", "Speed_Limit_50", "Priority_Road", "Give_Way", "Stop",
    "No_Entry", "Road_Work", "Traffic_Lights_Ahead", "Pedestrian_Crossing", "Roundabout", "No_Parking"
]

VAL_FRACTION = 0.20
RANDOM_SEED = 42

In [ ]:

# ── Step 1: Pool every existing image+label pair into one list ──────────────
all_pairs = []   # list of (image_path, label_path) tuples

for split_dir in (TRAIN_DIR, VALID_DIR):
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"
    if not img_dir.exists():
        continue
    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            all_pairs.append((img_path, lbl_path))

print(f"Pooled {len(all_pairs)} image+label pairs from existing dataset.")



Pooled 1926 image+label pairs from existing dataset.


In [ ]:
# ── Step 2: Shuffle and split 80/20 ─────────────────────────────────────────
random.seed(RANDOM_SEED)
random.shuffle(all_pairs)

split_point = int(len(all_pairs) * (1 - VAL_FRACTION))
new_train = all_pairs[:split_point]
new_valid = all_pairs[split_point:]

print(f"New split: {len(new_train)} train  |  {len(new_valid)} valid")

New split: 1540 train  |  386 valid


In [ ]:
import time

# ── Step 3: Move files into a temp staging area first ──────────────────────
# Note: Operations on Google Drive are slow because they are network-based.
staging = DATASET_DIR / "_staging"
if staging.exists():
    shutil.rmtree(staging)

print("Starting re-shuffling. This may take a few minutes on Google Drive...")

total_files = len(new_train) + len(new_valid)
processed = 0
last_update = time.time()

for split_name, pairs in [("train", new_train), ("valid", new_valid)]:
    (staging / split_name / "images").mkdir(parents=True, exist_ok=True)
    (staging / split_name / "labels").mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in pairs:
        if img_path.exists() and lbl_path.exists():
            # copy2 is safer on Drive to ensure metadata is preserved
            shutil.copy2(str(img_path), staging / split_name / "images" / img_path.name)
            shutil.copy2(str(lbl_path), staging / split_name / "labels" / lbl_path.name)

        processed += 1
        # Update progress every 5 seconds to avoid flooding output
        if time.time() - last_update > 5:
            print(f"Progress: {processed}/{total_files} pairs processed...")
            last_update = time.time()

print("Finalizing directory structure...")
# Wipe the old folders
shutil.rmtree(TRAIN_DIR, ignore_errors=True)
shutil.rmtree(VALID_DIR, ignore_errors=True)

# Move whole directories (much faster than individual files)
shutil.move(str(staging / "train"), str(TRAIN_DIR))
shutil.move(str(staging / "valid"), str(VALID_DIR))

# Clean up
if staging.exists():
    shutil.rmtree(staging)

print(f"Done! Successfully processed {total_files} pairs.")

Starting re-shuffling. This may take a few minutes on Google Drive...
Progress: 97/1926 pairs processed...
Progress: 185/1926 pairs processed...
Progress: 315/1926 pairs processed...
Progress: 420/1926 pairs processed...
Progress: 530/1926 pairs processed...
Progress: 596/1926 pairs processed...
Progress: 733/1926 pairs processed...
Progress: 842/1926 pairs processed...
Progress: 1001/1926 pairs processed...
Progress: 1078/1926 pairs processed...
Progress: 1204/1926 pairs processed...
Progress: 1288/1926 pairs processed...
Progress: 1443/1926 pairs processed...
Progress: 1504/1926 pairs processed...
Progress: 1598/1926 pairs processed...
Progress: 1706/1926 pairs processed...
Progress: 1794/1926 pairs processed...
Progress: 1894/1926 pairs processed...
Finalizing directory structure...
Done! Successfully processed 1926 pairs.


In [ ]:
# ── Step 4: Verify per-class counts in the new split ────────────────────────
print(f"\n── Repaired split — per-class instance counts ──────────────")

for split_name in ("train", "valid"):
    split_dir = DATASET_DIR / split_name
    img_count = len(list((split_dir / "images").glob("*")))
    lbl_count = len(list((split_dir / "labels").glob("*.txt")))
    print(f"\n[{split_name}]  images: {img_count}   labels: {lbl_count}")

    class_counts = Counter()
    for lbl_file in (split_dir / "labels").glob("*.txt"):
        for line in lbl_file.read_text().splitlines():
            if line.strip():
                class_counts[int(line.split()[0])] += 1

    # Dynamic range based on the actual length of your class list
    for class_id in range(len(OUR_CLASS_NAMES)):
        n = class_counts.get(class_id, 0)
        flag = "  [!]" if n < 1 else ""
        print(f"    {class_id:>2}  {OUR_CLASS_NAMES[class_id]:<22} {n:>5} instances{flag}")


── Repaired split — per-class instance counts ──────────────

[train]  images: 1495   labels: 1495
     0  Speed_Limit_30           253 instances
     1  Speed_Limit_50           212 instances
     2  Priority_Road            115 instances
     3  Give_Way                 343 instances
     4  Stop                     218 instances
     5  No_Entry                 193 instances
     6  Road_Work                 88 instances
     7  Traffic_Lights_Ahead      57 instances
     8  Pedestrian_Crossing      132 instances
     9  Roundabout               103 instances
    10  No_Parking               195 instances

[valid]  images: 382   labels: 382
     0  Speed_Limit_30            64 instances
     1  Speed_Limit_50            56 instances
     2  Priority_Road             25 instances
     3  Give_Way                  93 instances
     4  Stop                      42 instances
     5  No_Entry                  52 instances
     6  Road_Work                 26 instances
     7  Traffic_Li

In [ ]:
import yaml

# Ensure we use the persistent path
DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')
yaml_path = DATASET_DIR / "data.yaml"

# Define the configuration for YOLO
config = {
    "train": str(DATASET_DIR / "train/images"),
    "val": str(DATASET_DIR / "valid/images"),
    "nc": len(OUR_CLASS_NAMES),
    "names": OUR_CLASS_NAMES
}

# Write the config to data.yaml
with open(yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"\ndata.yaml updated with {len(OUR_CLASS_NAMES)} classes at: {yaml_path}")
print(yaml_path.read_text())

print("[✓] Setup complete. Your model will now train with 11 classes.")


data.yaml updated with 11 classes at: /content/drive/My Drive/TrafficSignProject/dataset/data.yaml
names:
- Speed_Limit_30
- Speed_Limit_50
- Priority_Road
- Give_Way
- Stop
- No_Entry
- Road_Work
- Traffic_Lights_Ahead
- Pedestrian_Crossing
- Roundabout
- No_Parking
nc: 11
train: /content/drive/My Drive/TrafficSignProject/dataset/train/images
val: /content/drive/My Drive/TrafficSignProject/dataset/valid/images

[✓] Setup complete. Your model will now train with 11 classes.


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import os

# Initialize the YOLO model.
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="/content/drive/MyDrive/traffic-sign-fyp/runs",
    name='gtsdb_merged_run',
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My Drive/TrafficSignProject/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install ultralytics -q

from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best.pt")

In [11]:
from pathlib import Path

DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')
results = model.predict(source=str(DATASET_DIR / "valid/images"), imgsz=640, conf=0.5, save=True)


image 1/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-103~info_bus_station-5.jpg: 640x640 1 No_Entry, 447.0ms
image 2/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-104~info_bus_station-6~info_crosswalk-8.jpg: 640x640 1 No_Entry, 487.9ms
image 3/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-105.jpg: 640x640 (no detections), 489.2ms
image 4/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-108~info_bus_station-8~info_crosswalk-9.jpg: 640x640 1 No_Entry, 532.7ms
image 5/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-113~mand_go_left-2.jpg: 640x640 1 No_Entry, 1561.4ms
image 6/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-12-13.jpg: 640x640 2 No_Entrys, 465.7ms
image 7/443 /content/drive/My Drive/TrafficSignProject/dataset/valid/images/forb_no_entry-130.jpg: 640x640 1 No_Entry, 439

In [3]:
from pathlib import Path

# Re-defining the path in case the runtime was restarted or variables were cleared
DATASET_DIR = Path('/content/drive/My Drive/TrafficSignProject/dataset')

# Set the model to evaluation mode for better on-device performance
model.eval()

model.export(
    format="litert", # Use the recommended 'litert' format
    quantize='int8', # Use 'int8' as a string for 8-bit integer quantization
    # Use the local variable to avoid NameError
    data=str(DATASET_DIR / "data.yaml"),
    fraction=0.3,
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,584,297 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 15, 33600) (5.4 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/drive/My Drive/TrafficSignProject/dataset/data.yaml'
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 15.6±5.1 MB/s, size: 76.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/My Drive/TrafficSignProject/dataset/valid/labels... 133 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 133/133 3.1it/s 42.5s
val: New cache created: /content/drive/M


LiteRT: starting export with litert_torch 0.9.1...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

...

(00:00) [ DONE] Write Model to 
/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite (+00:00)

LiteRT: applying static quantization (int8 weights + int8 activations)...


/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py:472: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


Model name: /content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite
Original model size: 10.43 MiB
Quantized model size: 2.97 MiB
Quantization Ratio: 0.29 (3.5x smaller)
Total time: 192.92 ms
LiteRT: export success ✅ 473.6s, saved as '/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite' (3.0 MB)

Export complete (475.1s)
Results saved to /content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite
Predict:         yolo predict task=detect model=/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite imgsz=1280 
Validate:        yolo val task=detect model=/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite imgsz=1280 data=/content/drive/My Drive/TrafficSignProject/dataset/data.yaml  
Visualize:       https://netron.app


PosixPath('/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-6/weights/best_int8.tflite')

In [4]:
from google.colab import files
files.download("/content/drive/MyDrive/traffic-sign-fyp/runs/gtsdb_merged_run-5/weights/best_saved_model/best_int8.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python3 --version

Python 3.12.13
